# Web Ingestion Pipeline: Static, Dynamic & Metadata Scraping

## 1. Overview
The quality of a Retrieval-Augmented Generation (RAG) system is strictly bounded by the fidelity of its document ingestion pipeline. Garbage in $\to$ garbage out.

### Ingestion Stages Covered:
1. **Static Web Scraping**: `requests`, `BeautifulSoup4`, and `robots.txt` compliance.
2. **Intelligent Content Extraction**: `trafilatura` for boilerplate removal and mini news corpus creation.
3. **Dynamic Web Scraping**: Playwright basics for JavaScript-rendered pages: render, scroll, and intercept the JSON behind the page.
4. **Metadata Extraction & Boilerplate Sanitization**: Using `extruct` for OpenGraph and JSON-LD structured metadata extraction.


---
# Part 1: Static Web Scraping (HTTP, robots.txt & BeautifulSoup)

In [ ]:
%pip install -q requests beautifulsoup4 lxml trafilatura rich tqdm

## 1. HTTP Basics — What Happens When You Fetch a Page

In [ ]:
import requests

URL = "https://news.ycombinator.com"
response = requests.get(URL, timeout=10)

print(f"Status code : {response.status_code}")
print(f"Content-Type: {response.headers.get('Content-Type')}")
print(f"Server      : {response.headers.get('Server', 'n/a')}")
print(f"Body size   : {len(response.content):,} bytes")
print(f"Encoding    : {response.encoding}")
print(f"\nFirst 300 chars of HTML:")
print(response.text[:300])

In [ ]:
# Always use a realistic User-Agent — many sites block Python's default
# and add headers that a real browser would send
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

response = requests.get(URL, headers=HEADERS, timeout=10)
print(f"Status: {response.status_code}  |  Size: {len(response.content):,} bytes")

In [ ]:
# Sessions are more efficient for multiple requests to the same host
# They automatically handle cookies and reuse TCP connections

session = requests.Session()
session.headers.update(HEADERS)

# Inspect redirect chain
resp = session.get("http://github.com", allow_redirects=True, timeout=10)
print("Redirect chain:")
for r in resp.history:
    print(f"  {r.status_code} → {r.url}")
print(f"Final: {resp.status_code} → {resp.url}")

## 2. robots.txt — Checking What We're Allowed to Scrape

Always check `robots.txt` before scraping. It's both a legal and ethical obligation.

In [ ]:
from urllib.robotparser import RobotFileParser
from urllib.parse import urlparse

def can_fetch(url: str, user_agent: str = "*") -> dict:
    """Check robots.txt before scraping a URL."""
    parsed = urlparse(url)
    robots_url = f"{parsed.scheme}://{parsed.netloc}/robots.txt"
    rp = RobotFileParser()
    rp.set_url(robots_url)
    try:
        rp.read()
        allowed = rp.can_fetch(user_agent, url)
        delay = rp.crawl_delay(user_agent)
        return {"url": url, "allowed": allowed, "crawl_delay": delay, "robots_url": robots_url}
    except Exception as e:
        return {"url": url, "allowed": True, "error": str(e)}

test_urls = [
    "https://news.ycombinator.com/",
    "https://en.wikipedia.org/wiki/Natural_language_processing",
    "https://arxiv.org/abs/1706.03762",
]

for url in test_urls:
    result = can_fetch(url)
    status = "ALLOWED" if result["allowed"] else "BLOCKED "
    delay = f" (crawl-delay: {result['crawl_delay']}s)" if result.get("crawl_delay") else ""
    print(f"{status}{delay}  →  {url}")

## 3. BeautifulSoup — Parsing HTML

In [ ]:
from bs4 import BeautifulSoup

resp = session.get("https://news.ycombinator.com", timeout=10)
soup = BeautifulSoup(resp.text, "lxml")

print(f"Page title : {soup.title.get_text()}")
print(f"Total links: {len(soup.find_all('a'))}") 
print(f"Total tags : {len(soup.find_all())}")

In [ ]:
# Extract Hacker News stories: title, URL, score, comment count
stories = []

for row in soup.select("tr.athing"):
    title_tag = row.select_one("span.titleline > a")
    if not title_tag:
        continue
    
    # sibling row has score and comments
    subrow = row.find_next_sibling("tr")
    score_tag = subrow.select_one("span.score") if subrow else None
    comment_tags = subrow.select("a") if subrow else []
    comment_tag = next((a for a in comment_tags if "comment" in a.text or "discuss" in a.text), None)
    
    stories.append({
        "title":    title_tag.get_text(strip=True),
        "url":      title_tag.get("href", ""),
        "score":    score_tag.get_text(strip=True) if score_tag else "0 points",
        "comments": comment_tag.get_text(strip=True) if comment_tag else "0",
    })

print(f"Stories scraped: {len(stories)}\n")
for s in stories[:5]:
    print(f"  [{s['score']:12s}] {s['title'][:70]}")
    print(f"  {s['url'][:80]}\n")

In [ ]:
# CSS selectors vs find_all — both approaches
wiki_resp = session.get(
    "https://en.wikipedia.org/wiki/Natural_language_processing", timeout=10
)
wiki_soup = BeautifulSoup(wiki_resp.text, "lxml")

# Method 1: CSS selectors (concise, familiar from web dev)
headings_css = wiki_soup.select("h2, h3")

# Method 2: find_all with attrs
headings_find = wiki_soup.find_all(["h2", "h3"])

print(f"Headings via CSS selector: {len(headings_css)}")
print(f"Headings via find_all:     {len(headings_find)}")

print("\nSection headings:")
for h in headings_css[:12]:
    # Wikipedia wraps edit buttons in headings — strip them
    text = h.get_text(" ", strip=True).split("[edit]")[0].strip()
    level = h.name  # h2 or h3
    indent = "  " if level == "h3" else ""
    print(f"  {indent}{level.upper()}: {text}")

In [ ]:
# Extract all paragraphs from the Wikipedia article
# Filter out short nav/footer paragraphs
content_div = wiki_soup.select_one("div#mw-content-text")
paragraphs = [
    p.get_text(" ", strip=True)
    for p in content_div.select("p")
    if len(p.get_text(strip=True)) > 100
]

print(f"Content paragraphs: {len(paragraphs)}")
print(f"\nFirst paragraph:\n{paragraphs[0]}")
print(f"\nSecond paragraph:\n{paragraphs[1]}")

In [ ]:
# Extract all external links with their anchor text
external_links = [
    {"text": a.get_text(strip=True), "href": a["href"]}
    for a in wiki_soup.select("a[href^='http']")
    if a.get_text(strip=True) and len(a.get_text(strip=True)) > 3
]

print(f"External links: {len(external_links)}")
for link in external_links[:8]:
    print(f"  '{link['text'][:40]:40s}' → {link['href'][:60]}")

## 4. Scraping a News Article — Full Extraction

In [ ]:
import re
from datetime import datetime

def scrape_article(url: str) -> dict:
    """Scrape a news article: title, author, date, body text, tags."""
    resp = session.get(url, timeout=15)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "lxml")

    # Title: try og:title → <title> tag → first <h1>
    og_title = soup.select_one('meta[property="og:title"]')
    h1 = soup.select_one("h1")
    title = (
        og_title["content"] if og_title
        else h1.get_text(strip=True) if h1
        else soup.title.get_text(strip=True) if soup.title
        else ""
    )

    # Published date: try common meta tags
    date_selectors = [
        'meta[property="article:published_time"]',
        'meta[name="pubdate"]',
        'time[datetime]',
    ]
    pub_date = ""
    for sel in date_selectors:
        tag = soup.select_one(sel)
        if tag:
            pub_date = tag.get("content") or tag.get("datetime", "")
            break

    # Author
    author_tag = soup.select_one('meta[name="author"], [rel="author"]')
    author = author_tag.get("content") or author_tag.get_text(strip=True) if author_tag else ""

    # Body: collect all <p> tags with substantial text
    body = " ".join(
        p.get_text(" ", strip=True)
        for p in soup.select("article p, .post-content p, .entry-content p, main p")
        if len(p.get_text(strip=True)) > 80
    )

    return {
        "url":        url,
        "title":      title,
        "author":     author,
        "published":  pub_date,
        "body":       body,
        "scraped_at": datetime.now().isoformat(),
        "word_count": len(body.split()),
    }

# Test on a Wikipedia article (reliable, no paywall)
article = scrape_article("https://en.wikipedia.org/wiki/Transformer_(deep_learning_architecture)")
print(f"Title      : {article['title']}")
print(f"Author     : {article['author']}")
print(f"Published  : {article['published']}")
print(f"Word count : {article['word_count']}")
print(f"\nBody preview:\n{article['body'][:400]}")

## 5. trafilatura — Intelligent Content Extraction

`trafilatura` uses machine learning to identify the main content of a page, removing navbars, ads, footers, and boilerplate.

In [ ]:
import trafilatura

TEST_URL = "https://en.wikipedia.org/wiki/Transformer_(deep_learning_architecture)"

# trafilatura can fetch and extract in one call
downloaded = trafilatura.fetch_url(TEST_URL)
extracted  = trafilatura.extract(downloaded)

print(f"Raw HTML size   : {len(downloaded):,} chars")
print(f"Extracted size  : {len(extracted or ''):,} chars")
print(f"Compression ratio: {len(extracted or '') / len(downloaded) * 100:.1f}%")
print(f"\n--- Extracted content (first 800 chars) ---")
print(extracted[:800] if extracted else "Extraction failed")

In [ ]:
# trafilatura: rich extraction with metadata
result = trafilatura.extract(
    downloaded,
    include_comments=False,
    include_tables=True,
    output_format="json",
    with_metadata=True,
    url=TEST_URL,
)

import json
data = json.loads(result)
print("Extracted metadata fields:")
for k, v in data.items():
    if k != "text":
        print(f"  {k:15s}: {str(v)[:80]}")
print(f"\nText length: {len(data.get('text', '')):,} chars")

In [ ]:
# Side-by-side comparison: raw HTML scraping vs trafilatura
def scrape_raw(url: str) -> str:
    """Naive extraction — all <p> text."""
    resp = session.get(url, timeout=10)
    soup = BeautifulSoup(resp.text, "lxml")
    return " ".join(p.get_text(" ", strip=True) for p in soup.find_all("p"))

raw_text   = scrape_raw(TEST_URL)
clean_text = trafilatura.extract(trafilatura.fetch_url(TEST_URL)) or ""

# Count sentences and unique words as proxy for signal quality
def text_stats(text: str, label: str):
    words = text.split()
    sentences = [s for s in re.split(r'[.!?]', text) if len(s.strip()) > 20]
    unique = len(set(w.lower() for w in words if w.isalpha()))
    print(f"{label}:")
    print(f"  Total chars    : {len(text):,}")
    print(f"  Total words    : {len(words):,}")
    print(f"  Unique words   : {unique:,}")
    print(f"  Sentences est. : {len(sentences):,}")

text_stats(raw_text, "Raw <p> scraping")
print()
text_stats(clean_text, "trafilatura")

## 6. Building a Mini News Corpus

Scrape multiple Wikipedia articles on NLP topics and save as a dataset.

In [ ]:
from pathlib import Path
import json
import time
from tqdm import tqdm


DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

NLP_TOPICS = [
    "https://en.wikipedia.org/wiki/Natural_language_processing",
    "https://en.wikipedia.org/wiki/Transformer_(deep_learning_architecture)",
    "https://en.wikipedia.org/wiki/BERT_(language_model)",
    "https://en.wikipedia.org/wiki/Word2vec",
    "https://en.wikipedia.org/wiki/Generative_pre-trained_transformer",
    "https://en.wikipedia.org/wiki/Attention_mechanism",
]

corpus = []
for url in tqdm(NLP_TOPICS, desc="Scraping articles"):
    try:
        downloaded = trafilatura.fetch_url(url)
        result_json = trafilatura.extract(
            downloaded,
            output_format="json",
            with_metadata=True,
            url=url,
        )
        if result_json:
            data = json.loads(result_json)
            corpus.append({
                "url":        url,
                "title":      data.get("title", ""),
                "text":       data.get("text", ""),
                "word_count": len(data.get("text", "").split()),
            })
            print(f"  {data.get('title', url)[:50]:50s} | {len(data.get('text','').split()):>5} words")
    except Exception as e:
        print(f"  {url}: {e}")
    time.sleep(0.5)

print(f"\nCorpus: {len(corpus)} articles, {sum(d['word_count'] for d in corpus):,} total words")

In [ ]:
# Save corpus as JSONL
corpus_path = DATA_DIR / "nlp_corpus.jsonl"
with open(corpus_path, "w") as f:
    for doc in corpus:
        f.write(json.dumps(doc) + "\n")

print(f"Saved: {corpus_path} ({corpus_path.stat().st_size / 1024:.1f} KB)")

# Quick stats
print("\nCorpus summary:")
for doc in corpus:
    print(f"  {doc['title'][:45]:45s} | {doc['word_count']:>5} words")

## 7. Exercise

Pick a topic and scrape at least 10 related articles:

1. Check `robots.txt` before starting
2. Use a `requests.Session` with realistic headers
3. Extract title, date, author, and body with BeautifulSoup
4. Compare raw extraction vs. `trafilatura` on 3 articles
5. Save your corpus as JSONL

**Stretch goal:** Add a `crawl_delay` to your scraper so it reads the delay from `robots.txt` and respects it.

In [ ]:
# Your code here
SEED_URLS = [
    # Add your URLs here
]

# Step 1: check robots.txt
# Step 2: create session with headers
# Step 3: scrape and extract
# Step 4: compare with trafilatura
# Step 5: save to JSONL

---
# Part 2: Dynamic Web Scraping with Playwright (JavaScript pages)

In [ ]:
%pip install -q playwright beautifulsoup4 lxml tqdm rich

In [ ]:
# Install browser binaries — only needed once
# This downloads Chromium (~150MB)
import subprocess
result = subprocess.run(
    ["python", "-m", "playwright", "install", "chromium"],
    capture_output=True, text=True
)
print(result.stdout[-200:] if result.stdout else "")
print(result.stderr[-200:] if result.stderr else "")
print("Browser install complete" if result.returncode == 0 else f"Error: {result.returncode}")

## 1. Why Static Scraping Fails on JS-Rendered Pages

In [ ]:
import requests
from bs4 import BeautifulSoup

# quotes.toscrape.com/js/ is a demo site that renders quotes via JavaScript
JS_URL = "https://quotes.toscrape.com/js/"

# Static request — gets the HTML shell before JS executes
resp = requests.get(JS_URL, timeout=10)
soup = BeautifulSoup(resp.text, "lxml")
static_quotes = soup.select(".quote .text")

print(f"Quotes found with static requests: {len(static_quotes)}")
print("(Expected 10 — JS hasn't run so we get 0 or the raw template)")

## 2. Playwright Basics — Sync API

In [ ]:
from playwright.async_api import async_playwright
from pathlib import Path

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

async with async_playwright() as p:
    browser = await p.chromium.launch(headless=True)
    page = await browser.new_page()

    await page.goto(JS_URL, wait_until="networkidle")

    # Now extract — JS has executed, DOM is populated
    quotes = await page.query_selector_all(".quote")
    print(f"Quotes found after JS execution: {len(quotes)}")

    for q in quotes[:3]:
        text_elem = await q.query_selector(".text")
        text = await text_elem.inner_text()
        author_elem = await q.query_selector(".author")
        author = await author_elem.inner_text()
        print(f"  {author:20s}: {text[:60]}")

    await browser.close()

In [ ]:
# Screenshots — great for debugging and visual verification
async with async_playwright() as p:
    browser = await p.chromium.launch(headless=True)
    page = await browser.new_page(viewport={"width": 1280, "height": 900})
    await page.goto(JS_URL, wait_until="networkidle")

    screenshot_path = DATA_DIR / "playwright_screenshot.png"
    await page.screenshot(path=str(screenshot_path), full_page=True)
    await browser.close()

from IPython.display import Image, display
display(Image(filename=str(screenshot_path), width=600))
print(f"Screenshot saved: {screenshot_path}")

In [ ]:
# Playwright context manager pattern — reusable browser + page
# Best practice: one browser, multiple pages (more efficient)
async with async_playwright() as p:
    browser = await p.chromium.launch(
        headless=True,
        args=["--no-sandbox", "--disable-dev-shm-usage"]
    )
    context = await browser.new_context(
        user_agent=(
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
        ),
        viewport={"width": 1280, "height": 900},
        locale="en-US",
    )
    page = await context.new_page()

    # Navigate and wait for a specific element
    await page.goto("https://quotes.toscrape.com/js/", wait_until="domcontentloaded")
    await page.wait_for_selector(".quote", timeout=5000)

    # Get full HTML after JS execution
    html = await page.content()
    print(f"HTML size after JS: {len(html):,} chars")
    
    # Parse with BeautifulSoup as normal
    soup = BeautifulSoup(html, "lxml")
    quotes = soup.select(".quote")
    print(f"Quotes parsed by BS4: {len(quotes)}")

    await browser.close()

## 3. Infinite Scroll — Extracting Lazy-Loaded Content

In [ ]:
# quotes.toscrape.com/scroll uses infinite scroll
SCROLL_URL = "https://quotes.toscrape.com/scroll"

async def scrape_infinite_scroll(url: str, max_scrolls: int = 5) -> list[dict]:
    """Scroll a page to trigger lazy loading, collecting all items."""
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page(viewport={"width": 1280, "height": 900})
        await page.goto(url, wait_until="domcontentloaded")
        await page.wait_for_selector(".quote")

        seen_count = 0
        for scroll_num in range(max_scrolls):
            # Scroll to bottom
            await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
            # Wait for new content to load
            await page.wait_for_timeout(800)

            current_count = await page.evaluate("document.querySelectorAll('.quote').length")
            print(f"  Scroll {scroll_num + 1}: {current_count} quotes loaded")

            if current_count == seen_count:
                print("  No new content — reached end")
                break
            seen_count = current_count

        # Extract all loaded quotes
        html = await page.content()
        await browser.close()

    soup = BeautifulSoup(html, "lxml")
    return [
        {
            "text":   q.select_one(".text").get_text(strip=True),
            "author": q.select_one(".author").get_text(strip=True),
            "tags":   [t.get_text(strip=True) for t in q.select(".tag")],
        }
        for q in soup.select(".quote")
    ]

quotes = await scrape_infinite_scroll(SCROLL_URL, max_scrolls=6)
print(f"\nTotal quotes collected: {len(quotes)}")
for q in quotes[:3]:
    print(f"  {q['author']:20s}: {q['text'][:60]}")

## 4. Intercepting Network Requests — Get the JSON Directly

In [ ]:
# Many SPAs fetch data via XHR/fetch — intercept those calls
# This is often cleaner than parsing HTML

api_responses = []

async def handle_response(response):
    """Capture JSON API responses."""
    if response.request.resource_type == "xhr" or "api" in response.url:
        try:
            content_type = response.headers.get("content-type", "")
            if "json" in content_type:
                body = await response.body()
                api_responses.append({
                    "url":    response.url,
                    "status": response.status,
                    "size":   len(body),
                })
        except Exception:
            pass

async with async_playwright() as p:
    browser = await p.chromium.launch(headless=True)
    page = await browser.new_page()
    page.on("response", handle_response)

    await page.goto("https://quotes.toscrape.com/scroll", wait_until="networkidle")
    await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
    await page.wait_for_timeout(2000)
    await browser.close()

if api_responses:
    print(f"JSON API calls intercepted: {len(api_responses)}")
    for r in api_responses:
        print(f"  {r['status']} | {r['size']:>6} bytes | {r['url'][:80]}")
else:
    print("No JSON API calls detected on this page")
    print("(This site uses server-rendered HTML, not a REST API)")

## 5. Exercise

Choose a JavaScript-rendered website (e.g., a job board, product listing, or social feed) and:

1. Show that `requests` fails to get the data
2. Use Playwright to load it properly
3. If the page uses infinite scroll or dynamic loading, collect multiple batches of items
4. Save as JSONL

**Stretch goal:** Use network interception to find the underlying API call and hit it directly with `requests` instead.

In [ ]:
# Your code here
TARGET_URL = ""  # your JS-rendered target

# Step 1: prove static scraping fails
# Step 2: Playwright scrape
# Step 3: infinite scroll / dynamic loading
# Step 4: save to JSONL

---
# Part 3: Metadata Extraction & Boilerplate Sanitization

In [ ]:
%pip install -q extruct w3lib

In [ ]:
import extruct
import requests
import pprint
from w3lib.html import get_base_url

In [ ]:
pp = pprint.PrettyPrinter(indent=2)
r = requests.get('https://cnn.com/')
base_url = get_base_url(r.text, r.url)
data = extruct.extract(r.text, base_url=base_url)

pp.pprint(data)

In [ ]:
import requests

def get_html(url):
    """Get raw HTML from a URL."""
    headers = {
        'Access-Control-Allow-Origin': '*',
        'Access-Control-Allow-Methods': 'GET',
        'Access-Control-Allow-Headers': 'Content-Type',
        'Access-Control-Max-Age': '3600',
        'User-Agent': 'Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:52.0) Gecko/20100101 Firefox/52.0'
    }
    req = requests.get(url, headers=headers)
    return req.text

In [ ]:
def get_metadata(html, url):
    """Fetch JSON-LD structured data."""
    metadata = extruct.extract(
        html,
        base_url=get_base_url(html, url),
        syntaxes=['json-ld'],
    )['json-ld'][0]
    return metadata

def scrape(url):
    """Parse structured data from a target page."""
    html = get_html(url)
    metadata = get_metadata(html, url)
    pprint.pprint(metadata, indent=2, width=150)
    return metadata

In [ ]:
scrape('https://cnn.com/')

## Summary & Key Takeaways — Web Ingestion Pipeline

1. **Static vs. Dynamic Selection**: Use HTTP + BeautifulSoup / Trafilatura for fast, lightweight static HTML extraction (~50x faster); reserve Playwright headless browsers for JavaScript-rendered SPAs, infinite scroll, and authenticated sessions.
2. **Polite & Compliant Crawling**: Respect `robots.txt`, configure realistic `User-Agent` headers, reuse HTTP sessions, and pace your requests to avoid throttling.
3. **Boilerplate Stripping**: Use `trafilatura` to isolate core article prose and automatically discard navigation bars, ads, sidebars, and footer clutter.
4. **Metadata Preservation**: Extract OpenGraph and JSON-LD structured metadata with `extruct` to enrich downstream vector chunks with timestamps, authorship, and document canonical URLs.
